In [1]:
# ── Imports ───────────────────────────────────────────────────────────────────
import glob
import json
import datetime
import statistics
from uuid import uuid4

import yaml
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, VectorParams, Distance, HnswConfigDiff
from fastembed import TextEmbedding
from chonkie import RecursiveChunker
import chonkie

from src.evaluator import Evaluator
from src.all_functionality import async_chat_wrapper

Evaluation functions defined.
Visualization function defined.


### Full notion

In [6]:
from pathlib import Path

docs_dir = Path("./rag_exp/notion_docs")
md_paths = sorted(docs_dir.glob("**/*.md"))

notion_docs = {}
for p in md_paths:
    with p.open("r", encoding="utf-8") as fh:
        notion_docs[p.stem] = fh.read()

print(f"Loaded {len(notion_docs)} markdown files from {docs_dir}")


Loaded 6 markdown files from rag_exp\notion_docs


In [8]:
notion_docs = {
    key: value for key, value in notion_docs.items() if "create" not in key
}
print(len(notion_docs))

3


In [9]:
testing_doc = notion_docs["block_reference"] 

In [21]:
import re
from chonkie import MarkdownChef, CodeChunker, RecursiveChunker

# Method 1: MarkdownChef
print("=" * 60)
print("Method 1: MarkdownChef")
print("=" * 60)
md_chef = MarkdownChef()
md_doc = md_chef.parse(testing_doc)
print(f"Document type: {type(md_doc)}")
print(f"Total chunks: {len(md_doc.chunks)}")
if md_doc.chunks:
    print(f"\nFirst chunk:\n{md_doc.chunks[0]}\n")

# Method 2: CodeChunker
print("=" * 60)
print("Method 2: CodeChunker")
print("=" * 60)
code_chunker = CodeChunker()
print(f"CodeChunker methods: {[m for m in dir(code_chunker) if not m.startswith('_')]}")
code_chunks = code_chunker.chunk(testing_doc)
print(f"Total chunks: {len(code_chunks)}")
if code_chunks:
    print(f"\nFirst chunk:\n{code_chunks[0]}\n")

# Method 3: RecursiveChunker
print("=" * 60)
print("Method 3: RecursiveChunker")
print("=" * 60)
recursive_chunker = RecursiveChunker()
recursive_chunks = recursive_chunker.chunk(testing_doc)
print(f"Total chunks: {len(recursive_chunks)}")
if recursive_chunks:
    print(f"\nFirst chunk:\n{recursive_chunks[0]}\n")


Method 1: MarkdownChef
Document type: <class 'chonkie.types.markdown.MarkdownDocument'>
Total chunks: 65

First chunk:
---
source_url: https://developers.notion.com/reference/block.md
category: major
description: A block object represents a piece of content within Notion. The API translates the headings, toggles, paragraphs, lists, media, and more that you can interact with in the Notion UI as different [block type objects](/reference/block#block-type-objects).
---

> ## Documentation Index
> Fetch the complete documentation index at: https://developers.notion.com/llms.txt
> Use this file to discover all available pages before exploring further.

# Block

> A block object represents a piece of content within Notion. The API translates the headings, toggles, paragraphs, lists, media, and more that you can interact with in the Notion UI as different [block type objects](/reference/block#block-type-objects).

For example, the following block object represents a `Heading 2` in the Notion U

In [43]:
def chunk_by_tags(document: str) -> list:
    # Chunk document by matching opening/closing XML-like tags (e.g. <Note>...</Note>)
    pattern = re.compile(r'<(?P<tag>[A-Za-z0-9_\-]+)[^>]*>.*?</(?P=tag)>', re.DOTALL)

    tag_chunks = []
    start = 0
    for m in pattern.finditer(document):
        if m.start() > start:
            tag_chunks.append({"tag": None, "text": document[start:m.start()]})
        tag_chunks.append({"tag": m.group("tag"), "text": m.group(0)})
        start = m.end()
    if start < len(document):
        tag_chunks.append({"tag": None, "text": document[start:]})

    # remove empty chunks
    tag_chunks = [c for c in tag_chunks if c["text"].strip()]
    return tag_chunks

tag_chunks = chunk_by_tags(testing_doc)

print(f"Total tag-based chunks: {len(tag_chunks)}")
for i, c in enumerate(tag_chunks[:6]):
    snippet = c["text"][:200].replace("\n", "\\n")
    print(f"{i}: tag={c['tag']!r}, len={len(c['text'])}, preview={snippet!r}")

Total tag-based chunks: 103
0: tag=None, len=885, preview='---\\nsource_url: https://developers.notion.com/reference/block.md\\ncategory: major\\ndescription: A block object represents a piece of content within Notion. The API translates the headings, toggles, para'
1: tag='CodeGroup', len=1084, preview='<CodeGroup>\\n  ```json Example Block Object expandable theme={null}\\n  {\\n  \t"object": "block",\\n  \t"id": "c02fc1d3-db8b-45c5-a222-27595b15aea7",\\n  \t"parent": {\\n  \t\t"type": "page_id",\\n  \t\t"page_id": "5983'
2: tag=None, len=123, preview='\\n\\nUse the [Retrieve block children](/reference/get-block-children) endpoint to list all of the blocks on a page.\\n\\n## Keys\\n\\n'
3: tag='Note', len=277, preview='<Note>\\n  Fields marked with an \\* are available to integrations with any capabilities. Other properties require read content capabilities in order to be returned from the Notion API. Consult the [inte'
4: tag=None, len=27649, preview='\\n\\n| Field              | Typ

In [61]:
# Global tracking variables for inspection
global_stats = {
    "total_code": 0,
    "total_images": 0,
    "total_tables": 0,
    "total_metadata": 0,
}

def process_tag_chunks(tag_chunks: list) -> list:
    """
    Process tag_chunks through MarkdownChef to extract nested chunks, tables, code, and images.
    Removes auxiliary fields like 'chunk_type' and 'source' from the produced chunks.
    """
    global global_stats

    processed_chunks = []
    md_chef = MarkdownChef()

    for tag_chunk in tag_chunks:
        original_tag = tag_chunk["tag"]
        text = tag_chunk["text"]

        if original_tag is not None:
            processed_chunks.append({
                "tag": original_tag,
                "text": text,
            })
            continue

        # Parse with MarkdownChef
        try:
            md_doc = md_chef.parse(text)
        except Exception:
            # If parsing fails, keep as single text chunk
            print(f"MarkdownChef parsing failed for chunk with tag={original_tag!r}, treating as plain text.")
            processed_chunks.append({
                "tag": original_tag,
                "text": text,
            })
            continue

        # Extract and track metadata
        if md_doc.metadata:
            global_stats["total_metadata"] += len(md_doc.metadata) if isinstance(md_doc.metadata, list) else 1

        # Process parsed chunks from MarkdownChef
        for chunk in md_doc.chunks:
            processed_chunks.append({
                "tag": "text",
                "text": chunk.text,
            })

        # Extract and track tables
        if md_doc.tables:
            global_stats["total_tables"] += len(md_doc.tables)
            for i, table in enumerate(md_doc.tables):
                processed_chunks.append({
                    "tag": "table",
                    "text": table.content,  # Convert table object to string
                    "table_index": i
                })

        # Extract and track code
        if md_doc.code:
            global_stats["total_code"] += len(md_doc.code)
            for i, code in enumerate(md_doc.code):
                processed_chunks.append({
                    "tag": 'CodeGroup',
                    "text": code.content,  # Convert code object to string
                    "code_index": i
                })

        # Extract and track images
        if md_doc.images:
            global_stats["total_images"] += len(md_doc.images)
            for i, image in enumerate(md_doc.images):
                processed_chunks.append({
                    "tag": "image",
                    "text": image.content,  # Convert image object to string
                    "image_index": i
                })

    return processed_chunks

# Apply function to tag_chunks
expanded_chunks = process_tag_chunks(tag_chunks)

print(f"Initial tag chunks: {len(tag_chunks)}")
print(f"Expanded chunks after MarkdownChef processing: {len(expanded_chunks)}")
print(f"\nGlobal statistics:")
print(f"  Total code blocks: {global_stats['total_code']}")
print(f"  Total tables: {global_stats['total_tables']}")
print(f"  Total images: {global_stats['total_images']}")
print(f"  Total metadata: {global_stats['total_metadata']}")

print(f"\nFirst 10 expanded chunks:")
for i, chunk in enumerate(expanded_chunks[:10]):
    snippet = chunk["text"][:100].replace("\n", "\\n")
    type_label = chunk['tag']
    print(f"{i}: tag={chunk['tag']!r}, type={type_label!r}, len={len(chunk['text'])}, preview={snippet!r}")

unique_tags = set(c['tag'] for c in expanded_chunks)
print(f"\nUnique tags in expanded chunks: {unique_tags}")

# find None tags
print(f"\nChunks with None tag:")
for i, chunk in enumerate(expanded_chunks):
    if chunk['tag'] is None:
        snippet = chunk["text"][:100].replace("\n", "\\n")
        print(f"{i}: tag={chunk['tag']!r}, len={len(chunk['text'])}, preview={snippet!r}")

Initial tag chunks: 103
Expanded chunks after MarkdownChef processing: 130

Global statistics:
  Total code blocks: 0
  Total tables: 26
  Total images: 0
  Total metadata: 0

First 10 expanded chunks:
0: tag='text', type='text', len=885, preview='---\\nsource_url: https://developers.notion.com/reference/block.md\\ncategory: major\\ndescription: A bloc'
1: tag='CodeGroup', type='CodeGroup', len=1084, preview='<CodeGroup>\\n  ```json Example Block Object expandable theme={null}\\n  {\\n  \t"object": "block",\\n  \t"id"'
2: tag='text', type='text', len=123, preview='\\n\\nUse the [Retrieve block children](/reference/get-block-children) endpoint to list all of the block'
3: tag='Note', type='Note', len=277, preview='<Note>\\n  Fields marked with an \\* are available to integrations with any capabilities. Other propert'
4: tag='text', type='text', len=1103, preview='\\n#### Block types that support child blocks\\n\\nSome block types contain nested blocks. The following b'
5: tag='table', t

In [65]:
def build_summary_prompt(prior_context, snippet):
    prompt = f"""
    You are an assistant helping to summarize and extract information from a markdown document. 
    The document has been chunked based on XML-like tags and further processed with MarkdownChef to identify nested structures, tables, code blocks, images, and metadata.

    Prior context:
    <context>{prior_context}</context>

    Current snippet:
    <snippet>{snippet}</snippet>

    Please provide a concise summary of the current snippet, highlighting any key information, code, tables, or images. 
    Max length is 70 words. Focus on the most important details and insights.
    
    If snippet is a table, then summarize the table structure and key data points. If snippet is code, summarize what the code does. If snippet is an image, describe the image if possible. If snippet contains metadata, summarize the metadata content.
    """
    return prompt.strip()

In [66]:
from src.all_functionality import async_chat_wrapper

async def ask_llm(prompt):
    # Placeholder for LLM interaction
    return await async_chat_wrapper(
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        model_size="gemma4"
    )

In [67]:
import asyncio
from typing import Dict, Any

async def summarize_chunks_with_concurrency(chunks: list, max_concurrent: int = 10) -> list:
    """
    For every table/CodeGroup chunk, collect prior context (previous text chunk)
    and ask LLM with summarization prompt. Add "summary" field to these chunks.
    Process in parallel with max_concurrent limit.
    """
    
    # Create a semaphore to limit concurrent requests
    semaphore = asyncio.Semaphore(max_concurrent)
    
    async def summarize_single(chunk_idx: int, chunk: Dict[str, Any]) -> None:
        """Summarize a single chunk if it's a table or CodeGroup"""
        if chunk["tag"] not in ["table", "CodeGroup"]:
            return
        
        # Find prior text chunk
        prior_context = ""
        for i in range(chunk_idx - 1, -1, -1):
            if chunks[i]["tag"] == "text":
                prior_context = chunks[i]["text"]
                break
        
        if not prior_context:
            prior_context = "(No prior text context found)"
        
        # Build prompt
        prompt = build_summary_prompt(prior_context, chunk["text"])
        
        # Call LLM with semaphore
        async with semaphore:
            try:
                response = await ask_llm(prompt)
                chunk["summary"] = response
                print(f"✓ Summarized chunk {chunk_idx} (tag: {chunk['tag']}, len: {len(chunk['text'])})")
            except Exception as e:
                chunk["summary"] = f"Error: {str(e)}"
                print(f"✗ Error summarizing chunk {chunk_idx}: {str(e)}")
    
    # Create tasks for all chunks
    tasks = [summarize_single(i, chunk) for i, chunk in enumerate(chunks)]
    
    # Run all tasks concurrently with semaphore limit
    await asyncio.gather(*tasks)
    
    return chunks

# Run summarization
print("Starting parallel summarization of table/CodeGroup chunks...")
print(f"Processing {len([c for c in expanded_chunks if c['tag'] in ['table', 'CodeGroup']])} chunks with max_concurrent=10")
print()

expanded_chunks = await summarize_chunks_with_concurrency(expanded_chunks, max_concurrent=10)

print("\nSummarization complete!")
print(f"Chunks with summaries: {len([c for c in expanded_chunks if 'summary' in c])}")

# Show samples
print("\nSample summaries:")
for i, chunk in enumerate(expanded_chunks):
    if "summary" in chunk:
        print(f"\n[Chunk {i}] Tag: {chunk['tag']}")
        print(f"Preview: {chunk['text'][:150].replace(chr(10), ' ')}...")
        print(f"Summary: {chunk['summary'][:200]}")
        if i >= 2:  # Show first 3 with summaries
            break


Starting parallel summarization of table/CodeGroup chunks...
Processing 63 chunks with max_concurrent=10



2026-03-14 23:45:46 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


[async_chat_wrapper] Model: gemma-3-4b-it, finish_reason: stop
✓ Summarized chunk 15 (tag: CodeGroup, len: 198)


2026-03-14 23:45:46 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
2026-03-14 23:45:46 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


[async_chat_wrapper] Model: gemma-3-4b-it, finish_reason: stop
✓ Summarized chunk 21 (tag: CodeGroup, len: 460)
[async_chat_wrapper] Model: gemma-3-4b-it, finish_reason: stop
✓ Summarized chunk 20 (tag: table, len: 2905)


...

[async_chat_wrapper] Model: gemma-3-4b-it, finish_reason: stop
✓ Summarized chunk 114 (tag: table, len: 3486)
[async_chat_wrapper] Model: gemma-3-4b-it, finish_reason: stop
✓ Summarized chunk 126 (tag: CodeGroup, len: 273)


2026-03-14 23:48:49 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
2026-03-14 23:48:49 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
2026-03-14 23:48:49 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


[async_chat_wrapper] Model: gemma-3-4b-it, finish_reason: stop
✓ Summarized chunk 111 (tag: table, len: 800)
[async_chat_wrapper] Model: gemma-3-4b-it, finish_reason: stop
✓ Summarized chunk 118 (tag: CodeGroup, len: 454)
[async_chat_wrapper] Model: gemma-3-4b-it, finish_reason: stop
✓ Summarized chunk 106 (tag: table, len: 1623)


2026-03-14 23:48:49 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


[async_chat_wrapper] Model: gemma-3-4b-it, finish_reason: stop
✓ Summarized chunk 78 (tag: CodeGroup, len: 364)

Summarization complete!
Chunks with summaries: 63

Sample summaries:

[Chunk 1] Tag: CodeGroup
Preview: <CodeGroup>   ```json Example Block Object expandable theme={null}   {   	"object": "block",   	"id": "c02fc1d3-db8b-45c5-a222-27595b15aea7",   	"pare...
Summary: This snippet details a JSON example of a block object in Notion. It represents a `heading_2` block containing the text "Lacinato kale" in green. The block has a specific ID, parent page ID, creation a

[Chunk 5] Tag: table
Preview: | Field              | Type                                                                    | Description                                          ...
Summary: This snippet describes the metadata associated with a block in a system. It outlines key properties like `object`, `id`, `parent`, `type`, `created_time`, `created_by`, `last_edited_time`, and `last_e


In [71]:
def build_final_tag(chunks):
    new_chunks = []
    for chunk in chunks:
        if chunk.get("summary"):
            name = "summary"
        else:
            name = "text"
            
        chunk["final"] = f"<{chunk['tag']}>{chunk[name]}</{chunk['tag']}>"
        new_chunks.append(chunk)
    return new_chunks

final_chunks = build_final_tag(expanded_chunks)

apply recursive chunker for all text chunks with a UPPERCASED local varibles CHUNK_SIZE

In [83]:
from qdrant_client.models import Distance, HnswConfigDiff, PointStruct
import secrets
import string

# ──────────────────────────────────────────────────────────────────────────────
# 1. Setup Qdrant client
# ──────────────────────────────────────────────────────────────────────────────

qdrant_client = QdrantClient(":memory:")
COLLECTION_NAME = "notion_chunks"

# ──────────────────────────────────────────────────────────────────────────────
# 2. Utility function: Generate 8-character UUID identifiers
# ──────────────────────────────────────────────────────────────────────────────

def generate_short_uuid(length: int = 8) -> str:
    """Generate a random alphanumeric identifier."""
    chars = string.ascii_letters + string.digits
    return ''.join(secrets.choice(chars) for _ in range(length))

# Create identifier mapping
chunk_ids = {}
for i, chunk in enumerate(expanded_chunks):
    chunk_ids[i] = generate_short_uuid()

print(f"Generated {len(chunk_ids)} unique identifiers")
print(f"Sample IDs: {list(chunk_ids.items())[:5]}")

Generated 130 unique identifiers
Sample IDs: [(0, 'gWcCWCpz'), (1, 'mestkszs'), (2, 'ZOcETl1O'), (3, 'STUOEI9O'), (4, 'SoeIFy0a')]


In [154]:
# ──────────────────────────────────────────────────────────────────────────────
# 4. Embed all chunks and prepare points for Qdrant
# ──────────────────────────────────────────────────────────────────────────────

embedding_model = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")

# ──────────────────────────────────────────────────────────────────────────────
# First pass: Collect IDs and embeddings for all chunks
# ──────────────────────────────────────────────────────────────────────────────

chunk_ids = {}
chunk_embeddings = {}

for idx, chunk in enumerate(expanded_chunks):
    chunk_id_str = generate_short_uuid()
    chunk_ids[idx] = chunk_id_str
    
    # Get final field or fall back to text
    embedding_text = chunk.get("final", chunk.get("text", ""))
    
    # Generate embedding
    embedding = list(embedding_model.embed(embedding_text))[0]
    chunk_embeddings[idx] = embedding

print(f"Generated {len(chunk_ids)} unique IDs and embeddings")
print(f"Vector dimension: {len(list(chunk_embeddings.values())[0])}")

# ──────────────────────────────────────────────────────────────────────────────
# Second pass: Build neighbor relationships and create Qdrant points
# ──────────────────────────────────────────────────────────────────────────────

n_neighbors = 1
points = []

for idx, chunk in enumerate(expanded_chunks):
    chunk_id_str = chunk_ids[idx]
    chunk_type = chunk.get("tag", "text")
    raw_text = chunk.get("text", "")
    
    # Build neighbor list (indices N before and after)
    neighbor_uuids = []
    for neighbor_idx in range(max(0, idx - n_neighbors), idx):
        neighbor_uuids.append(chunk_ids[neighbor_idx])
    for neighbor_idx in range(idx + 1, min(len(expanded_chunks), idx + n_neighbors + 1)):
        neighbor_uuids.append(chunk_ids[neighbor_idx])
    
    # Create payload with all chunk info
    payload = {
        "chunk_index": idx,
        "chunk_id": chunk_id_str,
        "tag": chunk_type,
        "text": chunk.get("final"), 
        "raw_text": raw_text,
        "neighbor_ids": neighbor_uuids,
    }
    
    # Create point for Qdrant
    point = PointStruct(
        id=int(chunk_id_str.encode().hex()[:16], 16) % (2**31),  # Convert to numeric ID
        vector=chunk_embeddings[idx],
        payload=payload
    )
    points.append((chunk_id_str, point))

print(f"Built neighbor relationships for {len(points)} chunks")
print(f"Sample point ID: {points[0][0]}, neighbors: {points[0][1].payload['neighbor_ids'][:3]}")


Generated 130 unique IDs and embeddings
Vector dimension: 384
Built neighbor relationships for 130 chunks
Sample point ID: Q7bRHtRk, neighbors: ['AvJwC71F']


In [155]:

# ──────────────────────────────────────────────────────────────────────────────
# 5. Create Qdrant collection with HNSW config
# ──────────────────────────────────────────────────────────────────────────────

vector_dim = len(points[0][1].vector)

qdrant_client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=vector_dim,
        distance=Distance.COSINE
    ),
    hnsw_config=HnswConfigDiff(
        m=16,
        ef_construct=200,
    )
)

print(f"Created collection '{COLLECTION_NAME}' with HNSW config")

# Upload points to Qdrant
points_to_upload = [point for _, point in points]
qdrant_client.upsert(
    collection_name=COLLECTION_NAME,
    points=points_to_upload
)

print(f"Uploaded {len(points_to_upload)} points to Qdrant")

# Create mapping from numeric ID back to string ID
id_mapping = {point.id: chunk_id_str for chunk_id_str, point in points}
print(f"ID mapping created: {len(id_mapping)} entries")


Created collection 'notion_chunks' with HNSW config
Uploaded 130 points to Qdrant
ID mapping created: 130 entries


C:\Users\Dmitry\AppData\Local\Temp\ipykernel_11068\1331627443.py:7: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant_client.recreate_collection(


In [156]:
def get_results(all_results):
    # Ensure combined field exists and tracking sets
    all_results.setdefault("combined", [])
    all_results.setdefault("combined_documents", [])
    seen_chunk_indices = set()
    id_to_idx = {v: k for k, v in chunk_ids.items()}

    for query_text in all_results["queries"]:
        transformed_query = query_text.lower().strip()
        query_embedding = list(embedding_model.embed(transformed_query))[0]

        search_results = qdrant_client.query_points(
            collection_name=COLLECTION_NAME,
            query=query_embedding,
            limit=all_results["limit_per_query"]
        ).points

        query_result = {
            "query": query_text,
            "transformed_query": transformed_query,
            "results": [],
            "neighbors": [] if all_results["use_neighbours"] else None,
        }

        for scored_point in search_results:
            payload = scored_point.payload or {}
            chunk_idx = payload.get("chunk_index")
            chunk_id = payload.get("chunk_id")
            text = payload.get("text")

            result_item = {
                "score": scored_point.score,
                "id": chunk_id,
                "chunk_index": chunk_idx,
                "tag": payload.get("tag"),
                "text": text,
                "raw_text": payload.get("raw_text"),
            }
            query_result["results"].append(result_item)

            # Add document to combined_documents and combined (unique by chunk_index)
            if chunk_idx is not None and chunk_idx not in seen_chunk_indices:
                all_results["combined_documents"].append(result_item)
                all_results["combined"].append(result_item)
                seen_chunk_indices.add(chunk_idx)

            # Collect and add neighbors (unique)
            if all_results["use_neighbours"]:
                neighbor_ids = payload.get("neighbor_ids", []) or []
                for neighbor_id in neighbor_ids:
                    # Convert string ID to numeric ID for Qdrant lookup
                    neighbor_numeric_id = int(neighbor_id.encode().hex()[:16], 16) % (2**31)
                    
                    try:
                        neighbor_points = qdrant_client.retrieve(
                            collection_name=COLLECTION_NAME,
                            ids=[neighbor_numeric_id]
                        )
                    except Exception:
                        continue
                    
                    if not neighbor_points:
                        continue
                    
                    neighbor_point = neighbor_points[0]
                    neighbor_payload = neighbor_point.payload or {}
                    neighbor_idx = neighbor_payload.get("chunk_index")
                    
                    if neighbor_idx is None or neighbor_idx in seen_chunk_indices:
                        continue

                    neighbor_item = {
                        "chunk_index": neighbor_idx,
                        "id": neighbor_id,
                        "tag": neighbor_payload.get("tag"),
                        "text": neighbor_payload.get("text", ""),
                        "raw_text": neighbor_payload.get("raw_text", ""),
                    }

                    # append to query_result neighbors and combined only if not already present
                    if not any(n["id"] == neighbor_id for n in (query_result["neighbors"] or [])):
                        query_result["neighbors"].append(neighbor_item)
                    all_results["combined"].append(neighbor_item)
                    seen_chunk_indices.add(neighbor_idx)

        all_results["query_results"].append(query_result)

    return all_results

In [190]:
# ──────────────────────────────────────────────────────────────────────────────
# 6. Query Qdrant with a list of queries and return combined results
# ──────────────────────────────────────────────────────────────────────────────

# Define queries to execute
queries = ["making a to-do block object"]
limit_per_query = 4
use_neighbours = True

all_results = {
    "queries": queries,
    "limit_per_query": limit_per_query,
    "use_neighbours": use_neighbours,
    "combined_documents": [],
    "query_results": []
}

all_results = get_results(all_results)

# print combined ids
print(list(set(r["id"] for r in all_results["combined"])))

['a4U1rF6y', '6rFS69B9', 'gbrhBj1U', 'p2yhkRuc', 'EBc8z72p', 'eN147O98', 'OZxKUVMb', 'bKaEiDPd', 'JtKd78Gb']


In [191]:
for doc in all_results["combined"]:
    print(f"Chunk index: {doc['chunk_index']}, ID: {doc['id']}, Tag: {doc['tag']}")
    if "score" in doc:
        print(f"Score: {doc['score']:.4f}")
    print(f"Text preview: {doc['text']}")

Chunk index: 113, ID: bKaEiDPd, Tag: text
Score: 0.8575
Text preview: <text>

### To do

To do block objects contain the following information within the `to_do` property:

</text>
Chunk index: 112, ID: eN147O98, Tag: CodeGroup
Text preview: <CodeGroup>The snippet presents a JSON example of a template block object. Specifically, it defines a template with a rich text element containing the instruction "Add a new to-do." The `template` property holds an array of rich text elements, indicating a potential for multiple template instructions within a larger template structure.</CodeGroup>
Chunk index: 114, ID: JtKd78Gb, Tag: table
Text preview: <table>This snippet describes the structure of a “To do” block object within a documentation system. It details the fields: `rich_text` (an array of rich text objects), `checked` (a boolean), `color` (an enumerated string), and `children` (an array of block objects). It provides a list of valid color options.</table>
Chunk index: 116, ID: 6rFS69B9, 

In [160]:
print(len(all_results["combined"]))

12


In [ ]:
from fastembed.rerank.cross_encoder import TextCrossEncoder

model = TextCrossEncoder(model_name='jinaai/jina-reranker-v1-turbo-en')

docs = [doc["raw_text"] for doc in all_results["combined"]]

results = model.rerank(queries[0], docs)
results = list(results)

In [188]:
# sort indices by value
sorted_results = [i for v, i in sorted([[v, i] for i, v in enumerate(list(results))], key=lambda x: x[0], reverse=True)]
for i in sorted_results[:5]:
    print(f"Score: {results[i]:.4f}, Text preview: {docs[i][:2000].replace(chr(10), ' ')}...")

Score: -0.2812, Text preview: | Field       | Type                                                 | Description                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 | | :---------- | :--------------------------------------------------- | :-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

- short to top
- dilluted This snippet describes the structure
- higher recall is great
- usually images, tables, code blocks are actually perfectly described by a paragraph above